In [1]:
import pickle
import sys
sys.path.append('/home/unix/vkrhk/EmmaEmb/EmmaEmb/analysis')
from constants import IMG_OUTPUT_PATH, EMB_SPACES

emb_space = EMB_SPACES[0]
with open(f'{IMG_OUTPUT_PATH}/performance/{emb_space[0]}_torch.pkl', 'rb') as f:
    results = pickle.load(f)
results

{'test_acc': 0.8505096750836734,
 'test_roc_auc': 0.6730626230958429,
 'test_mcc': 0.2488571223541272,
 'test_f1': 0.4098789183905582,
 'test_auprc': 0.39885644709051316,
 'class_acc_df':              class  accuracy
 0          BINDING  0.259273
 1  CRYPTIC-BINDING  0.151362
 2      NON-BINDING  0.913773}

### Subplot A

In [10]:
literature_performance_per_class = {
        "Cell.membrane": {
            "la": 0.81,
            "emma_cosine": 0.54,
            "emma_euclidean": 0.58,
            "label_size": 1067,
            "text_label": "Mem"
        },
        "Cytoplasm": {"la": 0.81,
                      "emma_cosine": 0.38,
                      "emma_euclidean": 0.39,
                      "label_size": 2180,
                      "text_label": "Cyt"},
        "Endoplasmic.reticulum": {
            "la": 0.70,
            "emma_cosine": 0.26,
            "emma_euclidean": 0.25,
            "label_size": 689,
            "text_label": "End"
        },
        "Golgi.apparatus": {
            "la": 0.33,
            "emma_cosine": 0.19,
            "emma_euclidean": 0.18,
            "label_size": 286,
            "text_label": "Gol"
        },
        "Lysosome/Vacuole": {
            "la": 0.12,
            "emma_cosine": 0.11,
            "emma_euclidean": 0.11,
            "label_size": 257,
            "text_label": "Lys"
        },
        "Mitochondrion": {
            "la": 0.88,
            "emma_cosine": 0.46,
            "emma_euclidean": 0.41,
            "label_size": 1208,
            "text_label": "Mit"
        },
        "Nucleus": {"la": 0.90,
                    "emma_cosine": 0.66,
                    "emma_euclidean": 0.62,
                    "label_size": 3235,
                    "text_label": "Nuc"},
        "Peroxisome": {
            "la": 0.17,
            "emma_cosine": 0.09,
            "emma_euclidean": 0.09,
            "label_size": 124,
            "text_label": "Per"
        },
        "Plastid": {"la": 0.92,
                    "emma_cosine": 0.70,
                    "emma_euclidean": 0.66,
                    "label_size": 605,
                    "text_label": "Pla"},
        "Extracellular": {
            "la": 0.95,
            "emma_cosine": 0.77,
            "emma_euclidean": 0.71,
            "label_size": 1580,
            "text_label": "Ext"
        },
    }

literature_performance_per_binding_class = {
    "Binding": {
        "label_size": 237344,
        "text_label": "RB",
        "emma_cosine": 0.6828982405285156,
        "la": 0.366534,
    },
    "Cryptic-Binding": {
        "label_size": 13263,
        "text_label": "CB",
        "emma_cosine": 0.09908768755183593,
        "la": 0.226092,
    },
    "Non-Binding": {
        "label_size": 1792157,
        "text_label": "NB",
        "emma_cosine": 0.955756889602864,
        "la": 0.728754,
    }

}

In [11]:
import pandas as pd
import plotly.colors as pc
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

distance_metric = "cosine"
df = pd.DataFrame(literature_performance_per_binding_class).T.reset_index()
df.columns = ["binding_class",
                "label_size", "text_label", "cosine", "la"]
df['binding_class_name'] = df[
    'binding_class'
    ].str.replace(".", " ")
df['label_size'] = np.sqrt(df['label_size'].to_numpy().astype(float))

df_sorted = df.sort_values(by="label_size", ascending=False).reset_index(
    drop=True
    )
num_classes = df_sorted['binding_class_name'].nunique()
# custom_greens = pc.sample_colorscale(
#     'Greens', [
#         0.2 + 0.7 * (i/(num_classes - 1)) for i in range(num_classes)
#         ]
# )
# color_map = dict(zip(df_sorted['binding_class_name'],
#                         custom_greens))
df_sorted['label_size'] = df_sorted['label_size'].astype(float)

fig_1b = px.scatter(
    df_sorted,
    x="la",
    y=distance_metric,
    color="binding_class_name",
#     color_discrete_map=color_map,
    size="label_size",
    text="text_label",
)
fig_1b.update_layout(
    template="plotly_white",
    font={"family": "Arial", "color": "black", "size": 20},
    width=600,
    height=600,
    showlegend=False,
    xaxis=dict(
        showgrid=False,
        linecolor='black',
        linewidth=3,
        title="Accuracy of ESM2 for distinguishing binding site type",
        ticks='outside',           # show ticks outside the axis
        tickwidth=2,               # width of the ticks
        tickcolor='black',         # color of the ticks
        ticklen=6,                 # length of the ticks
        range=[0, 1]
    ),
    yaxis=dict(
        showgrid=False,
        linecolor='black',
        linewidth=3,
        title="Mean KNN feature alignment score",
        ticks='outside',           # show ticks outside the axis
        tickwidth=2,               # width of the ticks
        tickcolor='black',         # color of the ticks
        ticklen=6,                 # length of the ticks 
        range=[0, 1]
    )
)

fig_1b.update_traces(textposition="bottom center")

fig_1b.add_shape(
    type="line",line_width=2, line_dash="dash", line_color="lightgrey",
    x0=0, y0=0, 
    x1=1, y1=1
)